RAG APPLICSTION USING CLOUD VECTORDB-typesense


In [29]:
import os
import typesense
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())

TYPESENSE_HOST = os.getenv("TYPESENSE_HOST")
TYPESENSE_API_KEY = os.getenv("TYPESENSE_API_KEY")

if not TYPESENSE_HOST or not TYPESENSE_API_KEY:
    raise ValueError("Set TYPESENSE_HOST and TYPESENSE_API_KEY in .env")

In [30]:
client = typesense.Client({
    'nodes': [{
        'host': TYPESENSE_HOST,
        'port': 443,
        'protocol': 'https'
    }],
    'api_key': TYPESENSE_API_KEY,
    'connection_timeout_seconds': 2
})

In [10]:
client

In [11]:
books_schema = {
  'name': 'books',
  'fields': [
    {'name': 'title', 'type': 'string'},
    {'name': 'authors', 'type': 'string[]', 'facet': True},
    {'name': 'publication_year', 'type': 'int32', 'facet': True},
    {'name': 'ratings_count', 'type': 'int32'},
    {'name': 'average_rating', 'type': 'float'}
  ],
  'default_sorting_field': 'ratings_count'
}
print(client.collections.create(books_schema))

{'created_at': 1783168382, 'curation_sets': [], 'default_sorting_field': 'ratings_count', 'enable_nested_fields': False, 'fields': [{'facet': False, 'index': True, 'infix': False, 'locale': '', 'name': 'title', 'optional': False, 'sort': False, 'stem': False, 'stem_dictionary': '', 'store': True, 'truncate_len': 100, 'type': 'string'}, {'facet': True, 'index': True, 'infix': False, 'locale': '', 'name': 'authors', 'optional': False, 'sort': False, 'stem': False, 'stem_dictionary': '', 'store': True, 'truncate_len': 100, 'type': 'string[]'}, {'facet': True, 'index': True, 'infix': False, 'locale': '', 'name': 'publication_year', 'optional': False, 'sort': True, 'stem': False, 'stem_dictionary': '', 'store': True, 'truncate_len': 100, 'type': 'int32'}, {'facet': False, 'index': True, 'infix': False, 'locale': '', 'name': 'ratings_count', 'optional': False, 'sort': True, 'stem': False, 'stem_dictionary': '', 'store': True, 'truncate_len': 100, 'type': 'int32'}, {'facet': False, 'index': T

In [18]:

with open('../data/books.jsonl', 'r', encoding='utf-8') as jsonl_file:
    data = jsonl_file.read()
    client.collections['books'].documents.import_(data)

In [19]:
search_parameters = {
    'q': 'Harry Potter',
    'query_by': 'title,authors',
    'sort_by': 'ratings_count:desc'
 }
client.collections['books'].documents.search(search_parameters)

{'facet_counts': [],
 'found': 17,
 'hits': [{'document': {'authors': ['J.K. Rowling', ' Mary GrandPré'],
    'average_rating': 4.44,
    'id': '2',
    'image_url': 'https://images.gr-assets.com/books/1474154022m/3.jpg',
    'publication_year': 1997,
    'ratings_count': 4602479,
    'title': "Harry Potter and the Philosopher's Stone"},
   'highlight': {'title': {'matched_tokens': ['Harry', 'Potter'],
     'snippet': "<mark>Harry</mark> <mark>Potter</mark> and the Philosopher's Stone"}},
   'highlights': [{'field': 'title',
     'matched_tokens': ['Harry', 'Potter'],
     'snippet': "<mark>Harry</mark> <mark>Potter</mark> and the Philosopher's Stone"}],
   'text_match': 1157451471441102969,
   'text_match_info': {'best_field_score': '2211897868289',
    'best_field_weight': 15,
    'fields_matched': 1,
    'num_tokens_dropped': 0,
    'score': '1157451471441102969',
    'tokens_matched': 2,
    'typo_prefix_score': 0}},
  {'document': {'authors': ['J.K. Rowling', ' Mary GrandPré', ' R

In [21]:
### Langchain + Typsense + Groq LLM + RAG Application

from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import Typesense
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_groq import ChatGroq

In [22]:
import os
os.environ["GROQ_API_KEY"] = ""

In [31]:
docsearch=Typesense.from_documents(
    docs,
    embeddings,
    typesense_client_params={
        "host": TYPESENSE_HOST,  # Use xxx.a1.typesense.net for Typesense Cloud
        "port": "443",  # Use 443 for Typesense Cloud
        "protocol": "https",  # Use https for Typesense Cloud
        "typesense_api_key": TYPESENSE_API_KEY,
        "typesense_collection_name": "lang-chain"
    },
    
)

In [32]:
docsearch=Typesense.from_documents(
    docs,
    embeddings,
    typesense_client_params={
        "host": TYPESENSE_HOST,  # Use xxx.a1.typesense.net for Typesense Cloud
        "port": "443",  # Use 443 for Typesense Cloud
        "protocol": "https",  # Use https for Typesense Cloud
        "typesense_api_key": TYPESENSE_API_KEY,
        "typesense_collection_name": "lang-chain"
    },
    
)

In [25]:
query = "What is machine learning?"
found_docs = docsearch.similarity_search(query)
print(found_docs[0].page_content)

Machine Learning Basics

Machine learning (ML) is a branch of artificial intelligence (AI) that enables computers to learn patterns from data and make predictions or decisions without being explicitly programmed for every task. Instead of relying on fixed rules, machine learning algorithms improve their performance as they are exposed to more data.

Machine learning is widely used in industries such as healthcare, finance, e-commerce, cybersecurity, and autonomous systems. Common applications include fraud detection, recommendation systems, language translation, image recognition, speech processing, and predictive analytics.

Types of Machine Learning

1. Supervised Learning
Supervised learning uses labeled datasets where the correct output is already known. The model learns the relationship between inputs and outputs and can predict results for unseen data.

Examples:
- Email spam detection
- House price prediction
- Image classification
- Disease diagnosis


In [26]:
### Retriever
retriever = docsearch.as_retriever()
retriever

VectorStoreRetriever(tags=['Typesense', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.typesense.Typesense object at 0x00000197BA348FB0>, search_kwargs={})

In [28]:
query = "unsupervised learning indepth explanation"
retriever.invoke(query)[0]

Document(metadata={'source': '../data/text_files/machine_learning.txt'}, page_content='Examples:\n- Email spam detection\n- House price prediction\n- Image classification\n- Disease diagnosis\n\nPopular Algorithms:\n- Linear Regression\n- Logistic Regression\n- Decision Trees\n- Random Forest\n- Support Vector Machines\n- Neural Networks\n\n2. Unsupervised Learning\nUnsupervised learning works with unlabeled data. The objective is to discover hidden structures, relationships, or patterns within the data.\n\nExamples:\n- Customer segmentation\n- Market basket analysis\n- Topic modeling\n- Anomaly detection\n\nPopular Algorithms:\n- K-Means Clustering\n- DBSCAN\n- Hierarchical Clustering\n- Principal Component Analysis (PCA)\n\n3. Reinforcement Learning\nReinforcement learning trains an agent by interacting with an environment. The agent receives rewards for desirable actions and penalties for undesirable ones. Over time, it learns an optimal strategy to maximize cumulative rewards.\n\nA